<h1 style="text-align: center;">  Deployment & DevOps Notes</h1>
<h4 style="text-align: center;">Akshay Sehgal | www.akshaysehgal.com | 2021</h4>


# Django API setup

### Virtual environment setup


- Create a new `main_folder`
- Create a new virtual environment inside the `main_folder`

```python
python -m venv virtualenv
```

- Deactivate the current base environment `conda deactivate`
- Go to the folder for `main_folder/virtualenv/bin` and run `source activate` to activate the venv

- Once activated, install dependencies via terminal

```python
pip install django
pip install pandas
#etc...
```

### Project and App setup

- Go to `main_folder` in the terminal and create a django project

```python
django-admin startproject myproject
```

- Go into the `main_folder/myproject` in the terminal where the `manage.py` file exists and create a django app

```python
django-admin startapp myapp
```

Now, you should have the following file structure - 

```python
main_folder
|__ virtualenv
|__ myproject
    |__ manage.py        #<-- main file to run the app
    |__ db.sqlite3
    |__ myapp
        |__ views.py     #<-- connects the requests to the function calls
        |__ #etc...
    |__ myproject
        |__ settings.py  #<-- Contains the directory of Installed Apps
        |__ urls.py      #<-- Contains the URLs that need to be accessed for functions in views.py
        |__ #etc..
```

### Pycharm Setup

- Open Pycharm and Open the project by browsing `main_folder/myproject/`
- Next: 
    - In Pycharm go to `manage.py`, there should be an option to configure python interpreter on top **OR**
    - In bottom right of the Pycharm window, there should be an option to select interpreter
- Click on that and select `Add python interpreter`
- Select `Existing environment` then from the `...` browse button, navigate to the `main_folder/virtualenv/python3` and select it
- Wait for Pycharm to configure the intepreter

### Running the App

- In Pycharm, open `settings.py` and in the list of `INSTALLED_APPS` add the app name `myapp`.
- In Pycharm, open `views.py` add the following test code - 

```python
from django.http import HttpResponse

def index(request):
        return HttpResponse("Hello World!")
        
```

- In Pycharm, open `urls.py` and in the list of `urlpatterns` add the code to import the `myapp` and connect the index function to a url - 

```python
from myapp import views               #<-- add this

urlpatterns = [
    path('admin/', admin.site.urls),
    path('',views.index)              #<-- add this
]
```

- In Pycharm, open `manage.py`, right click on the code and `Run` it.
- Once done, on top-left of Pycharm, click on `manage` dropdown and `edit configurations`
- In the window, in the `PARAMETERS` field add `runserver` and rerun `manage.py` 
- You sould see the server running and the link to the localhost `http://127.0.0.1:8000/`
- Clicking this should show you the "Hello World!" message on the browser. 

### Handling requests and application logic


- Create a new python file `logic.py` inside `main_folder/myproject/myapp/`.
- Add your custom function `function(param)` to this file, along with all the necessary `imports`
- Once added, you can add call this function inside a function call in `views.py` and add the `views.py` function to `url.py` against a url.

> **NOTE:**
> 1. The goal of `logic.py` is to store `logic.function`. 
> 2. The goal of `views.py` is to store `views.functionCall` which calls the `logic.function`. 
> 3. The goal of `urls.py` is to connect the `views.functionCall` to an appropriate url.


**In `logic.py` file**, add the function that takes user input parameters - 

---
```python 
#LOGIC.PY
def function(param):
    output = ...
    return output
```
---

**In `views.py` file**, import the `myapp.logic.function` and then call it inside another function. For handling the request parameter, you can use `request.GET.get("param",'default')` or `request.POST.get("param",'default')` for get/post API calls respectively. For returning results, you can return either as `HttpResponse` or `JsonResponse`.

---
```python
#VIEWS.PY
from django.http import HttpResponse, JsonResponse
from myapp.logic import function

def functionCall(request):
    param = request.GET.get("param",'default')
    result = function(param)
    return HttpResponse(result)
```
---

**In `urls.py` file**, you can map the `myapp.views.functionCall` to `http://127.0.0.1:8000/getresults` url - 

---
```python
#URLS.PY
from django.urls import path
from myapp.views import functionCall

urlpatterns = [
    path('admin/', admin.site.urls),
    path('getresults', functionCall),   #<------
]
```
---

Hit the url `http://127.0.0.1:8000/getresults?param=value` (for GET call)

> **GET call** - Hit the url directly as `http://127.0.0.1:8000/getresults?param1=value1&param2=value2`. Or, use POSTMAN to add paramter keys and values to a GET request. 
>
> **POST call** - Hit the url `http://127.0.0.1:8000/getresults` but send the params in the body of the request as form-data keys and values.

----------------

# AWS EC2 Deployment

0. Create an AWS account
1. Login to AWS and click on `Services > Compute > EC2`
2. Follow the steps and launch an EC2 server with Ubuntu (preferrable in free tier)
3. Generate a .pem file and save it in a proper folder
4. Go to the instance dashboard and wait for the server to show status as running
5. Once done, you can login to the server via SSH

### Security setup (for opening ports)

1. After selecting the current running EC2 instance, go to `Network & Security > Security Groups` (from the menu on the RHS).
2. Next select the security group which is mentioned in the Instance dashboard infront of the specific EC2 instance
3. In the table below, select `Inbound` and add rule - `Custom TCP`, set port to 8000, select `anywhere` and add `0.0.0.0/0`
4. Save
5. Now you should be able to access the port 8000 at `0.0.0.0:8000`

### SSH login

1. First, cd into the folder with the .pem file, and use the command `chmod 400 pemfilename.pem`
2. Next, copy the `Public IPv4 DNS` from the Instance Dashboard on AWS
3. Use the following command in terminal to login from the folder containing the .pem file

```
ssh -i pemfilename.pem ubuntu@ec2-xxxxxxxxxxxxxxxx.compute.amazonaws.com
```

### Server setup

Once logged in, run the following - 

```
$ sudo apt update
$ sudo apt install python3-pip
$ pip3 install django
$ pip3 install requests
$ ...etc...
```

### Code transfer

1. **IMPORTANT:** In the `setting.py` of the django project, add the **Public IPv4 address** (from Instance Dashboard) to `ALLOWED_HOSTS`, or simply use `ALLOWED_HOSTS = ['*']` to allow all hosts
2. Install Filezilla
3. Select Site-manager in top left corner and select Add site
4. Add the following details 
  - Protocol = SFTP
  - Host = **Public IPv4 DNS** (from Instance Dashboard)
  - Logon Type = Key file
  - User = ubuntu
  - Key File = path/to/pemfilename.pem
5. Connect
6. Once connected, drag and drop the complete project folder into the Ubuntu folder and wait for file transfer

### Running the server

1. Once done, log back into the EC2 instance from the terminal ssh command, navigate to the project folder containing `manage.py`
2. Run the following command to run the server - 

```python
#Run server on localhost port 8000
python3 manage.py runserver 0.0.0.0:8000

#OR for running the server in the background (only if moving to pre-production!)
nohup python3 manage.py runserver 0.0.0.0:8000 

```
3. This would run the server on port `8000`.
4. Now you can publically access your api from `ip-address:8000` (**Public IPv4 address** from Instance Dashboard)

-----------------

# Ngrok Setup

Ngrok is a simple to use tool to help with api tunneling localhost to web and for setup of webhooks for deployment by forwarding http->https

1. Signup for ngrok on `https://ngrok.com/` and copy your authtoken
2. Next, install snap and use it to download and install ngrok on Ubuntu server (Mac has a separate downloader)

```
$ sudo apt install snap
$ sudo snap install ngrok
```

3. Once installed, setup authentication to ensure Ngrok session doesn't expire

```
$ ngrok authtoken <your Auth Token>
```

4. Finally, run the following command to enable tunneling the exposed localhost 8000 port - 

```
$ ngrok http 8000
```

5. You can then copy the https url which ngrok is forwarding the localhost port to and publically call it over https or use it for webhooks